# 01 — Baseline QLoRA Fine-tuning

Establish a QLoRA fine-tuning baseline on `meta-llama/Llama-3.2-3B-Instruct` before any pruning, R-tuning, or unlearning.

**Sequence**

1. Setup & config
2. Data loading (interactive MinIO picker)
3. Model loading with Unsloth (4-bit) + QLoRA adapters
4. Training (SFTTrainer)
5. Evaluation — perplexity on eval set
6. Inference test — all mental-health prompt batteries
7. Save model via MLflow + GGUF export

Logs to MLflow under experiment `burnit-bg-experiments`, run tagged `experiment=baseline`.


In [1]:
# ###### Colab bootstrap ######
# On Colab the [experiments] extras pulls both requirements_package.txt and
# requirements_experiments.txt in one pip command — single source of truth lives
# in setup.py's extras_require. Then bootstrap() loads Colab Secrets into
# os.environ and brings up Tailscale so *.ts.net hostnames are reachable.
# Locally bootstrap() just loads .env and checks the tailnet — no installs.
#
# Required Colab Secrets (key icon → Add new secret → toggle "Notebook access"):
#   TAILSCALE_AUTHKEY   – from https://login.tailscale.com/admin/settings/keys
#   MLFLOW_TRACKING_URI, MLFLOW_EXPERIMENT_NAME, MLFLOW_TRACKING_INSECURE_TLS
#   MINIO_ENDPOINT / MINIO_ACCESS_KEY / MINIO_SECRET_KEY / MINIO_SECURE
#   HF_TOKEN
import os, subprocess, sys
IN_COLAB = "COLAB_GPU" in os.environ or "google.colab" in sys.modules
if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "burnit_bg[experiments] @ git+https://github.com/kirilyotov/BurnIT-BG.git",
    ])

from utils.colab import bootstrap
bootstrap()


[set_env] local: loaded /home/kiril/AI/BurnIT-BG/.env
[colab] not in Colab — skipped GitHub install + Tailscale.


{'in_colab': False,
 'secrets_loaded': True,
 'package_installed': False,
 'requirements_installed': False,
 'tailscale_connected': True}

In [2]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"Device: {props.name}")
    print(f"VRAM:   {props.total_memory / 1024**3:.2f} GB")
    print(f"Compute capability: {props.major}.{props.minor}")


CUDA available: False


## 1. Setup & config

In [3]:
import sys, os
from pathlib import Path

REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'data_platform').exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from data_platform.common import set_env
from data_platform.storage import MinioStorage

from experiments.shared.mlflow_utils import setup_run, log_responses, stage, log_training_curves
from experiments.shared.inference_utils import (
    run_full_test_battery,
    TEST_PROMPTS_IN_DOMAIN, TEST_PROMPTS_OUT_OF_DOMAIN, TEST_PROMPTS_EDGE,
    format_prompt,
)
from experiments.shared.model_utils import (
    DEFAULT_MODEL_NAME, load_model_unsloth, apply_qlora, count_trainable_params,
    cuda_device_info,
)
from experiments.shared.eval_utils import compute_perplexity, benchmark_speed, vram_snapshot
from experiments.shared.dataset_utils import (
    load_alpaca_dataset, dataset_statistics, alpaca_to_prompt,
)
from experiments.shared.dataset_browser import list_datasets, pick_dataset, download_dataset, resolve


In [4]:
set_env(quiet=True)
tracking, tags, run_name = setup_run(
    experiment='baseline',
    model=DEFAULT_MODEL_NAME,
    stage="experiment",
    mlflow_experiment_name='burnit-bg-experiments',
)
print(f"run_name = {run_name}")
print(f"tags     = {tags}")
print(f"machine  = {cuda_device_info()}")


TrackingError: Failed to configure MLflow: API request to https://k3s-acer-f5-master.tail1e4f6a.ts.net/mlflow/api/2.0/mlflow/experiments/get-by-name failed with timeout exception HTTPSConnectionPool(host='k3s-acer-f5-master.tail1e4f6a.ts.net', port=443): Max retries exceeded with url: /mlflow/api/2.0/mlflow/experiments/get-by-name?experiment_name=TestLLMExperiment (Caused by ConnectTimeoutError(<HTTPSConnection(host='k3s-acer-f5-master.tail1e4f6a.ts.net', port=443) at 0x713eecde87d0>, 'Connection to k3s-acer-f5-master.tail1e4f6a.ts.net timed out. (connect timeout=120)')). To increase the timeout, set the environment variable MLFLOW_HTTP_REQUEST_TIMEOUT (default: 120) to a larger value.

## 2. Data loading

Picks a dataset from MinIO via `resolve()`. Set `DATASET_PREFIX` env var to skip the prompt, or pass `auto=True`.

In [ ]:
# ###### Pick a dataset from MinIO (or fall back to local data_prep/) ######
# Set DATASET_PREFIX in .env / Colab secrets to skip the picker, e.g.
#   DATASET_PREFIX=data_prep/processed/mental-health
# Or pass `auto=True` below for non-interactive runs.

DEFAULT_PREFIX = os.getenv("DATASET_PREFIX", "data_prep/processed")
try:
    local_dataset_dir = resolve(prefix=DEFAULT_PREFIX, auto=False)
    TRAIN_PATH = local_dataset_dir / "train.jsonl"
    EVAL_PATH  = local_dataset_dir / "eval.jsonl"
except (FileNotFoundError, Exception) as exc:
    print(f"[data] MinIO lookup failed ({exc}); falling back to local data_prep/processed/")
    TRAIN_PATH = REPO_ROOT / "data_prep/processed/train.jsonl"
    EVAL_PATH  = REPO_ROOT / "data_prep/processed/eval.jsonl"

train_records = list(load_alpaca_dataset(TRAIN_PATH))
eval_records  = list(load_alpaca_dataset(EVAL_PATH))
train_stats = dataset_statistics(train_records)
eval_stats  = dataset_statistics(eval_records)
print(f"train: {len(train_records)} rows  eval: {len(eval_records)} rows")
print(train_stats)


In [ ]:
# Quick category distribution chart
import pandas as pd, matplotlib.pyplot as plt
cat_counts = pd.Series(train_stats["by_category"]).sort_values(ascending=False)
ax = cat_counts.plot(kind="bar", figsize=(8, 3.5), title="Train: examples per category")
plt.tight_layout(); plt.show()


## 3. Model loading

In [ ]:
HF_TOKEN = os.getenv("HF_TOKEN")
MAX_SEQ_LEN = 2048

with stage(tracking, "load_model"):
    model, tokenizer = load_model_unsloth(
        DEFAULT_MODEL_NAME,
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=True,
        token=HF_TOKEN,
    )
    model = apply_qlora(model, r=16, lora_alpha=32)
    params = count_trainable_params(model)
    print(params)


## 4. Training

In [ ]:
from datasets import Dataset

def _format(record):
    record["text"] = alpaca_to_prompt(record, eos_token=tokenizer.eos_token)
    return record

train_ds = Dataset.from_list(train_records).map(_format)
eval_ds  = Dataset.from_list(eval_records).map(_format)


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

OUTPUT_DIR = REPO_ROOT / "tmp/experiments/baseline"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy="epoch",
    bf16=True,
    optim="adamw_8bit",
    report_to=[],
    lr_scheduler_type="cosine",
    max_grad_norm=1.0,
)

with tracking.run(run_name=run_name, tags=tags):
    tracking.log_params({
        **{f"data.{k}": v for k, v in train_stats.items() if not isinstance(v, dict)},
        **{f"model.{k}": v for k, v in params.items()},
        **{f"train.{k}": getattr(training_args, k) for k in (
            "num_train_epochs","per_device_train_batch_size","gradient_accumulation_steps",
            "learning_rate","warmup_ratio","lr_scheduler_type","max_grad_norm",
        )},
        "max_seq_length": MAX_SEQ_LEN,
    })

    with stage(tracking, "train"):
        trainer = SFTTrainer(
            model=model,
            tokenizer=tokenizer,
            train_dataset=train_ds,
            eval_dataset=eval_ds,
            dataset_text_field="text",
            max_seq_length=MAX_SEQ_LEN,
            args=training_args,
        )
        trainer_stats = trainer.train()

    history_steps = [h["step"] for h in trainer.state.log_history if "loss" in h]
    history_loss  = [h["loss"]  for h in trainer.state.log_history if "loss" in h]
    history_lr    = [h.get("learning_rate", 0.0) for h in trainer.state.log_history if "loss" in h]
    if history_steps:
        log_training_curves(tracking, steps=history_steps, losses=history_loss, learning_rates=history_lr, title="baseline")
    tracking.log_metrics({"final_train_loss": float(history_loss[-1]) if history_loss else 0.0})

    with stage(tracking, "evaluate"):
        ppl = compute_perplexity(model, tokenizer, [r["output"] for r in eval_records[:64]])
        tracking.log_metrics({"eval_perplexity": float(ppl)})
        print(f"eval perplexity = {ppl:.3f}")
        tracking.log_metrics({f"vram.{k}": v for k, v in vram_snapshot().items()})

    # Test prompts
    batteries = run_full_test_battery((model, tokenizer), max_new_tokens=256)
    log_responses(tracking, experiment="baseline",
                  model_checkpoint=str(OUTPUT_DIR), **batteries)
    for k, v in batteries.items():
        print(f"-- {k} --")
        for entry in v[:2]:
            print(f"  Q: {entry['prompt'][:80]}\n  A: {entry['response'][:200]}\n")

    # Save via MLflow — artifact + registered model version
    with stage(tracking, "save_model"):
        try:
            tracking.log_model(
                model,
                flavor="transformers",
                artifact_path="model",
                registered_model_name="burnit-bg-baseline",
            )
            print("[save] model logged via MLflow")
        except Exception as exc:
            print(f"[save] log_model failed: {exc}")

    # Optional GGUF export
    with stage(tracking, "gguf_export"):
        try:
            from experiments.shared.model_utils import export_gguf
            gguf_path = export_gguf(model, tokenizer, OUTPUT_DIR / "gguf", quantization="q4_k_m")
            tracking.save_data(gguf_path, artifact_path="gguf")
            print(f"[save] GGUF logged: {gguf_path}")
        except Exception as exc:
            print(f"[save] GGUF export skipped: {exc}")

    tracking.log_hardware(step=1)


## Next steps

* Compare this run in MLflow with later experiments by filtering tag `experiment=baseline`.
* Continue to `02_layer_pruning.ipynb` — it loads the baseline model from MLflow as its starting point.
